# Análisis Exploratorio Gráfico (GEDA)

## Base de datos: Consumo de agua, Ciudad de México

***

En la práctica anterior aprendimos a **conocer** los datos: cuántas variables hay, qué tipos son, información general de cada una, y valores faltantes existen. Ahora vamos a **verlos**.

El **GEDA** (*Graphical Exploratory Data Analysis*) nos permite detectar cosas que los números solos no muestran: distribuciones sesgadas, valores atípicos, patrones entre grupos, relaciones entre variables.

### Lo que aprenderemos en este notebook

1. Qué tipo de gráfico es el correcto según el tipo de variable
2. Qué hace que un gráfico sea fácil (o difícil) de leer
3. Cómo construir cada tipo de gráfico paso a paso
4. Cómo leer e interpretar lo que muestra cada gráfico

### Regla de oro antes de graficar

> **Primero pregunta, luego grafica.**  
> Cada gráfico debe responder una pregunta concreta. Si no sabes qué pregunta estás respondiendo, el gráfico probablemente no va a decir nada útil.


## ¿Qué gráfico usar?

La elección del gráfico depende de **qué tipo de variables** estás analizando (tabla no exahustiva):

| Pregunta | Variables involucradas | Gráfico recomendado |
|---|---|---|
| ¿Cómo se distribuye una variable? | 1 numérica | Histograma, boxplot |
| ¿Cuántos hay en cada categoría? | 1 categórica | Barras |
| ¿Varía una numérica entre categorías? | 1 numérica + 1 categórica | Boxplot por grupo, barras con promedio |
| ¿Existe relación entre dos numéricas? | 2 numéricas | Dispersión (scatter) |
| ¿Cómo cambia algo en el tiempo? | 1 numérica + 1 temporal | Línea |
| ¿Qué proporción tiene cada categoría? | 1 categórica | Barras (**¡nunca pay!**) |
| ¿Dónde están los datos geográficamente? | latitud + longitud | Mapa de puntos |


## 0. Importar librerías

Además de `pandas` y `numpy` para manejar datos, usamos las siguientes librerías para graficar:

- **`matplotlib`**: la base de casi toda visualización en Python. Da control total pero es más verbosa.
- **`seaborn`**: construida sobre matplotlib, tiene gráficos estadísticos más elaborados con menos código.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo base para todas las gráficas del notebook
# 'whitegrid' agrega líneas horizontales de referencia — ayudan a leer valores
sns.set_theme(style='whitegrid', palette='crest')

# Tamaño por defecto de las figuras (ancho x alto en pulgadas)
plt.rcParams['figure.figsize'] = (10, 5)

# Fuente más grande para que se lea bien en pantalla y en proyector
plt.rcParams['font.size'] = 12

### Cargar y preparar los datos

Retomamos el DataFrame limpio del laboratorio anterior. Aquí repetimos los pasos de limpieza para que este notebook funcione de forma independiente.

In [ ]:
# Cargamos los datos
df = pd.read_csv('https://raw.githubusercontent.com/ElenaVillano/eda_geda_conagua/refs/heads/master/data_consumo_agua_updated.csv')

# --- Hacemos algunas cosas de limpieza ---

# Estandarizamos texto a minúsculas
for col in ['alcaldia', 'colonia', 'indice_des','nomgeo']:
    df[col] = df[col].str.lower().str.strip()

# Bimestre como texto (es una categoría, no un número)
df['bimestre'] = df['bimestre'].astype(str)

# Fecha de registro es una variable temporal por lo que también la convertimos
df['fecha_registro'] = pd.to_datetime(df['fecha_registro'], errors='coerce')

print(f'Base lista: {df.shape[0]:,} filas × {df.shape[1]} columnas')
df.head(3)

## 1. Histograma ¿Cómo se distribuye una variable numérica?

**¿Cuándo usarlo?** Cuando quieres ver la *forma* de la distribución de una sola variable numérica: si es simétrica, sesgada, si tiene varios picos, etc.

**¿Cómo funciona?** Divide el rango de valores en intervalos (*bins*) y cuenta cuántas observaciones caen en cada uno.

### 1.1 Primer intento (sin ajustes)

In [ ]:
df.describe()

In [ ]:
# plt.hist() es la función básica de matplotlib para histogramas
# Solo necesita un array de valores
plt.hist(df['consumo_total'],
          bins=100,
         )
plt.show()

**¿Qué está mal con este gráfico?**
- No tiene título ni etiquetas en los ejes → no sabemos qué estamos viendo
- Casi todo el peso está en la primera barra → hay valores extremos que comprimen el gráfico
- Solo 10 bins (el valor por defecto) → la distribución se ve borrosa



### 1.2 Versión mejorada:

> *Anatomía de un histograma bien hecho*

Un buen histograma debe tener:
- **Título** que diga claramente qué se está mostrando
- **Etiqueta en el eje X** con el nombre de la variable y sus unidades
- **Etiqueta en el eje Y** (normalmente "Frecuencia" o "Número de registros")
- **Número de bins adecuado**: muy pocos bins ocultan la forma real; demasiados crean ruido

In [ ]:
# Primero calculamos el percentil 95 para recortar los valores extremos
# Esto no elimina los datos del DataFrame, solo los excluye del gráfico
p95 = df['consumo_total'].quantile(0.99)
datos_graf = df[df['consumo_total'] <= p95]['consumo_total']

In [ ]:
# --- Construimos el gráfico ---
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    datos_graf,
    bins=300,             #### IMPORTANTE Más bins = más detalle en la forma
    color='green',
    edgecolor='white',   # Borde blanco entre barras: más fácil de leer
    linewidth=0.5
)

# Elementos obligatorios de toda gráfica
ax.set_title('Distribución del consumo total de agua por toma\n(se excluye el 5% con mayor consumo)', fontsize=13)
ax.set_xlabel('Consumo total bimestral (m³)', fontsize=11)
ax.set_ylabel('Número de registros', fontsize=11)

# Línea vertical en la mediana — ayuda a ver si la distribución es simétrica
mediana = datos_graf.median()
ax.axvline(mediana, color='tomato', linestyle='--', linewidth=1.5, label=f'Mediana: {mediana:.0f} m³')
ax.legend()

# Nota sobre los datos excluidos
ax.text(0.98, 0.95,
        f'Nota: se muestran {len(datos_graf):,} de {len(df):,} registros',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=9, color='gray')

plt.tight_layout()
plt.show()

print(f'Mediana:     {mediana:,.0f} m³')
print(f'Media:       {datos_graf.mean():,.0f} m³  ← mayor que la mediana = distribución sesgada a la derecha')

#### ¿Qué nos dice este histograma?

- La distribución está **sesgada a la derecha**: la mayoría de las tomas tiene consumo bajo, pero hay algunas con consumo muy alto que jalan la media hacia arriba.
- Cuando la **media > mediana**, siempre hay sesgo positivo. Eso significa que el promedio no es el mejor resumen de esta variable.
- La **mediana** (línea roja) es más representativa del consumo "típico" que la media.

> **Regla práctica:** Si media y mediana difieren mucho, usa la mediana para resumir la variable. Si son similares, puedes usar cualquiera.

### 1.3 ¿Qué pasa si no recortamos los outliers?

In [ ]:
# Comparación: con y sin outliers, con mismos parámetros
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sin recorte
axes[0].hist(df['consumo_total'], bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_title('Con todos los datos\n(incluyendo valores extremos)')
axes[0].set_xlabel('Consumo total (m³)')
axes[0].set_ylabel('Número de registros')

# Con recorte al percentil 95
axes[1].hist(datos_graf, bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
axes[1].set_title('Sin el 5% con mayor consumo\n(percentil 95 como límite)')
axes[1].set_xlabel('Consumo total (m³)')
axes[1].set_ylabel('Número de registros')

plt.suptitle('Efecto de los valores extremos en el histograma', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

> **Lección importante:** Los valores extremos pueden hacer que un histograma muestre casi toda la distribución comprimida en una sola barra. Recortar visualmente (sin eliminar los datos) permite ver la distribución real del grueso de la población.
>
> Siempre menciona en el gráfico si hiciste este ajuste, como lo hicimos en el título.

## 2. Boxplot: Distribución y outliers de forma compacta

**¿Cuándo usarlo?** Cuando quieres comparar la distribución de una variable numérica **entre grupos** (categorías). Es más compacto que múltiples histogramas.

**¿Cómo leerlo?**

```
          │   ← Valor máximo (sin contar outliers)
          │
    ┌─────┤   ← Cuartil 75 (Q3): el 75% de los datos está abajo de esta línea
    │     │
    │─────│   ← Mediana (Q2): el valor central
    │     │
    └─────┤   ← Cuartil 25 (Q1): el 25% de los datos está abajo de esta línea
          │
          │   ← Valor mínimo (sin contar outliers)
          
       ∘  ∘   ← Puntos fuera: outliers
```

La **altura de la caja** (Q3 - Q1) se llama **rango intercuartílico (IQR)**. Mientras más alta, más dispersos están los datos centrales.

### 2.1 Boxplot simple: distribución de consumo_total

In [ ]:
# Trabajamos con datos recortados para que el boxplot sea legible
## ¿Qué pasa si le cambias al 95?
quantile_data = df['consumo_total'].quantile(0.99)

# Aquí estás subseleccionando los datos
df_box = df[df['consumo_total'] <= quantile_data].copy()

# Configuaciones para el gráfico bonito
fig, ax = plt.subplots(figsize=(10, 5))

# Boxplot
sns.boxplot(
    data=df_box,
    x='consumo_total',   # Variable numérica en el eje X = boxplot horizontal
                         # Qué pasa si pones x en lugar de x
    color='steelblue',        # Qué pasa si colocas otro color
    ax=ax
)

ax.set_title('Distribución del consumo total de agua por toma\n(sin el 1% con mayor consumo)', fontsize=13)
ax.set_xlabel('Consumo total (m³)', fontsize=11)
ax.set_yticks([])  # Quitamos el eje Y porque no tiene información útil aquí

plt.tight_layout()
plt.show()

### 2.2 Boxplot por grupos: ¿varía el consumo según el índice de desarrollo?

Aquí es donde el boxplot muestra su verdadero valor: comparar distribuciones entre categorías de un solo vistazo.

In [ ]:
# Definimos el orden lógico de las categorías (de mayor a menor desarrollo)
# Esto es importante: sin orden, seaborn las pone en orden alfabético, que no tiene sentido aquí
orden_indice = ['alto', 'medio', 'bajo', 'popular']

# Configuaciones para el gráfico bonito
fig, ax = plt.subplots(figsize=(11, 6))

# Boxplot
sns.boxplot(
    data=df_box,
    x='consumo_total',
    y='indice_des',
    order=orden_indice,    # Especificamos el orden manualmente
    palette='colorblind',       # Paleta de colores
    hue='indice_des',      # Es para especificar las categorías
    ax=ax
)

ax.set_title('Consumo total de agua según índice de desarrollo\n(sin el 1% con mayor consumo)', fontsize=13)
ax.set_xlabel('Consumo total (m³)', fontsize=11)
ax.set_ylabel('Índice de desarrollo', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
df['indice_des'].value_counts()

**¿Qué nos dice este gráfico?**
- Las zonas de índice **alto** tienen mayor consumo promedio y mayor dispersión.
- Las zonas de índice **medio** y **bajo** tienen distribuciones más compactas y centradas en valores más bajos, y son más similares.
- Las zonas de índice **popolar** tienen una distribución con menor consumo.
- Los **outliers** (puntos a la derecha) aparecen en todos los niveles, pero son más pronunciados en zonas de mayor desarrollo.

> ¿Cuántos datos hay cada distribución, recuerdas? ¿Cuál es el código para ver el número de observaciones por categoría?

### 2.3 Boxplot por bimestre: ¿cambia el consumo en el tiempo?

In [ ]:
# Creamos etiquetas más descriptivas para los bimestres
etiquetas_bim = {'1': 'B1\nEne–Feb', '2': 'B2\nMar–Abr', '3': 'B3\nMay–Jun'}
df_box['bimestre_label'] = df_box['bimestre'].map(etiquetas_bim)

fig, ax = plt.subplots(figsize=(10, 5))

sns.boxplot(
    data=df_box,
    y='bimestre_label', # ¿Qué pasa si el bimestre va en el eje y?
    x='consumo_total',
    order=['B1\nEne–Feb', 'B2\nMar–Abr', 'B3\nMay–Jun'],
    palette='Set2',
    hue='bimestre_label',
    legend=False,  # ¿Qué pasa si le pones True?
    ax=ax
)

ax.set_title('Distribución del consumo total por bimestre', fontsize=13)
ax.set_xlabel('Bimestre', fontsize=11)
ax.set_ylabel('Consumo total (m³)', fontsize=11)

plt.tight_layout()
plt.show()

## 3. Gráfica de barras: comparar categorías

**¿Cuándo usarlo?** Para comparar una cantidad (conteo, suma, promedio) entre categorías.

**Reglas para gráficas de barras legibles:**
1. **Ordenar las barras** de mayor a menor (o por algún criterio lógico). Las barras sin orden obligan al lector a buscar con los ojos.
2. **Preferir barras horizontales** cuando hay muchas categorías con nombres largos — evita rotar etiquetas.
3. **No empezar el eje Y en un valor distinto de 0** — distorsiona la percepción de las diferencias.
4. **Una pregunta = una gráfica**. No mezcles demasiadas variables en una sola barra.

### 3.1 ¿Cuántos registros tiene cada alcaldía? (conteo simple)

In [ ]:
# Contamos registros por alcaldía y ordenamos de mayor a menor
conteo_alcaldia = df['alcaldia'].value_counts()  # value_counts() ya ordena de mayor a menor

# ¿Seleccionar nomgeo o alcaldía?

fig, ax = plt.subplots(figsize=(10, 7))

# Barras horizontales: más legibles cuando los nombres son largos
# .sort_values() aquí pone las barras de menor a mayor (de abajo arriba)
# así la alcaldía con más registros queda arriba — más natural para leer
conteo_alcaldia.sort_values().plot(
    kind='barh',
    ax=ax,
    color='pink',
    edgecolor='white'
)

ax.set_title('Número de registros por alcaldía', fontsize=13)
ax.set_xlabel('Número de registros', fontsize=11)
ax.set_ylabel('')  # Las alcaldías ya son las etiquetas del eje Y, no necesitamos título

# Añadimos el valor al final de cada barra — elimina la necesidad de leer el eje
for i, (valor, nombre) in enumerate(zip(conteo_alcaldia.sort_values(), conteo_alcaldia.sort_values().index)):
    ax.text(valor + 50, i, f'{valor:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

### 3.2 ¿Cuál es el consumo promedio por alcaldía?

Ojo: esto es diferente al conteo. Aquí agrupamos por alcaldía y calculamos el promedio.

In [ ]:
prom_alcaldia = df.groupby('alcaldia')['consumo_total'].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))

prom_alcaldia.plot(
    kind='barh',
    ax=ax,
    color='coral',
    edgecolor='white'
)

ax.set_title('Consumo promedio de agua por alcaldía\n(promedio de todas las tomas, 2019)', fontsize=13)
ax.set_xlabel('Consumo promedio bimestral (m³)', fontsize=11)
ax.set_ylabel('')

# Línea de referencia en el promedio general
prom_general = df['consumo_total'].mean()
ax.axvline(prom_general, color='navy', linestyle='--', linewidth=1.2,
           label=f'Promedio general: {prom_general:.0f} m³')
ax.legend()

# Añadir el número de registros por alcaldía al final de cada barra
conteo_alcaldia_sorted = df['alcaldia'].value_counts().reindex(prom_alcaldia.index)
for i, (valor_promedio, num_registros) in enumerate(zip(prom_alcaldia, conteo_alcaldia_sorted)):
    ax.text(valor_promedio + 50, i, f'({num_registros:,} registros)', va='center', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

> **¿Qué dice la línea punteada?** El promedio general divide las alcaldías en dos grupos: las que están por encima del promedio de la ciudad y las que están por debajo. Es una referencia útil para contextualizar.

### 3.3 Barras apiladas: Consumo por tipo de uso y alcaldía

Las barras apiladas muestran tanto el **total** como la **composición** de ese total. Son útiles cuando quieres ver proporciones dentro de cada categoría.

In [ ]:
# Calculamos el consumo total por alcaldía para cada tipo de uso
consumo_tipo = df.groupby('alcaldia')[['consumo_total_dom',
                                       'consumo_total_no_dom',
                                       'consumo_total_mixto']].sum()

# Renombramos para que la leyenda sea más legible
consumo_tipo.columns = ['Doméstico', 'No doméstico', 'Mixto']

# Ordenamos por consumo doméstico total (de mayor a menor)
consumo_tipo = consumo_tipo.sort_values('Doméstico', ascending=True)

fig, ax = plt.subplots(figsize=(11, 8))

consumo_tipo.plot(
    kind='barh',
    stacked=True,           # stacked=True activa el apilamiento, que pasa si le pones False
    ax=ax,
    color=['steelblue', 'coral', 'mediumseagreen'],
    edgecolor='white',
    linewidth=0.4
)

ax.set_title('Consumo total de agua por tipo de uso y alcaldía\n(suma del año 2019, tres bimestres)', fontsize=13)
ax.set_xlabel('Consumo total acumulado (m³)', fontsize=11)
ax.set_ylabel('')
ax.legend(title='Tipo de uso', bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout()
plt.show()

**¿Cuándo NO usar barras apiladas?**  
Cuando hay más de 4-5 categorías apiladas, se vuelve muy difícil comparar los segmentos intermedios porque no parten del mismo punto. En ese caso, es mejor usar barras agrupadas o gráficas separadas.

## 4. Gráfica de dispersión (scatter): Relación entre dos variables numéricas

**¿Cuándo usarlo?** Para explorar si existe una **relación** (correlación) entre dos variables numéricas. Cada punto es una observación.

**¿Qué buscamos?**
- ¿Los puntos forman una nube con tendencia? → hay correlación
- ¿Los puntos están dispersos sin patrón? → no hay correlación clara
- ¿Hay grupos visibles? → puede haber subpoblaciones

### 4.1 ¿Se relaciona el consumo doméstico con el no doméstico?

In [ ]:
# Para el scatter filtramos outliers extremos para que la nube de puntos sea legible
p95_dom    = df['consumo_total_dom'].quantile(0.95)
p95_nodom  = df['consumo_total_no_dom'].quantile(0.95)
df_scatter = df[(df['consumo_total_dom'] <= p95_dom) &
                (df['consumo_total_no_dom'] <= p95_nodom)]

fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(
    df_scatter['consumo_total_dom'],
    df_scatter['consumo_total_no_dom'],
    alpha=0.15,      # alpha: transparencia. Con muchos puntos, la superposición
                     # hace que las zonas densas se vean más oscuras — eso es información
    s=8,             # s: tamaño del punto
    color='steelblue'
)

ax.set_title('Relación entre consumo doméstico y no doméstico\npor toma de agua', fontsize=13)
ax.set_xlabel('Consumo doméstico (m³)', fontsize=11)
ax.set_ylabel('Consumo no doméstico (m³)', fontsize=11)

# Nota con el número de puntos graficados
ax.text(0.98, 0.02, f'n = {len(df_scatter):,} registros',
        transform=ax.transAxes, ha='right', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

### 4.2 Scatter con color por índice de desarrollo — Relación entre dos numéricas con una categórica

In [ ]:
# Paleta de colores para cada nivel del índice de desarrollo
paleta = {
    'popular':    '#1f77b4', # Azul
    'medio':   '#ff7f0e', # Naranja
    'bajo': '#2ca02c', # Verde
    'alto':    '#d62728'  # Rojo
}


fig, ax = plt.subplots(figsize=(10, 7))

# Graficamos cada nivel de indice_des por separado para aplicar el color y la leyenda
for nivel in ['medio', 'bajo', 'popular','alto']:
    subset_des = df_scatter[df_scatter['indice_des'] == nivel]
    ax.scatter(
        subset_des['consumo_total_dom'],
        subset_des['consumo_total_no_dom'],
        c=paleta[nivel], # Definidos en 'paleta'
        s=8,
        alpha=0.3,
        label=nivel.capitalize()
    )

ax.set_title('Relación entre consumo doméstico y no doméstico\npor toma de agua, coloreado por índice de desarrollo', fontsize=13)
ax.set_xlabel('Consumo doméstico (m³)', fontsize=11)
ax.set_ylabel('Consumo no doméstico (m³)', fontsize=11)
ax.legend(title='Índice de desarrollo')

# Nota con el número de puntos graficados
ax.text(0.98, 0.02, f'n = {len(df_scatter):,} registros', # Using df_scatter to maintain consistency with previous plot
        transform=ax.transAxes, ha='right', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

> ¿Qué te dice el gráfico, qué conclusiones puedes obtener? ¿Consideras que se puede mejorar el gráfico?

### 4.3 Mejorando la visualización con facetas

Relación por índice de desarrollo

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Definimos el orden lógico para los títulos de las facetas y para la leyenda
orden_indice = ['alto', 'medio', 'bajo', 'popular']

# Usamos seaborn.lmplot para crear un scatter plot con facetas por indice_des y línea de regresión
g = sns.lmplot(
    data=df_scatter,
    x='consumo_total_dom',
    y='consumo_total_no_dom',
    col='indice_des',
    col_wrap=2,
    hue='indice_des',
    palette=paleta,
    col_order=orden_indice,
    height=4, aspect=1.2,
    scatter_kws={'s': 8, 'alpha': 0.3}, # Arguments for the scatter points
    line_kws={'color': 'black', 'lw': 1} # Customize regression line color and width
)

# Añadimos títulos y etiquetas para cada faceta
g.set_titles('Índice de desarrollo: {col_name}')
g.set_xlabels('Consumo doméstico (m³)')
g.set_ylabels('Consumo no doméstico (m³)')

plt.suptitle('Relación entre consumo doméstico y no doméstico por toma de agua\nseparado por índice de desarrollo', y=1.05, fontsize=14) # Título general

plt.show()

Esta visualización hace más sencilla la lectura del gráfico, sin embargo, ¿qué pregunta está respondiendo esta información? ¿Te ayuda a obtener más informaicón de la base?

### 5.1 ¿En qué alcaldías hay más valores faltantes en consumo mixto?

In [ ]:
# Calculamos el % de faltantes en consumo_total_mixto por alcaldía
# Primero: contamos totales y faltantes por alcaldía
totales    = df.groupby('alcaldia')['consumo_total_mixto'].count().rename('con_valor')
faltantes  = df.groupby('alcaldia')['consumo_total_mixto'].apply(lambda x: x.isnull().sum()).rename('faltantes')
total_filas = df.groupby('alcaldia')['consumo_total_mixto'].size().rename('total')

resumen_miss = pd.concat([total_filas, faltantes], axis=1)
resumen_miss['pct_faltante'] = (resumen_miss['faltantes'] / resumen_miss['total'] * 100).round(1)
resumen_miss = resumen_miss.sort_values('pct_faltante', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))

# Coloreamos por severidad: rojo = mucho faltante
colores = ['tomato' if p > 50 else 'steelblue' for p in resumen_miss['pct_faltante']]

ax.barh(resumen_miss.index, resumen_miss['pct_faltante'], color=colores, edgecolor='white')

# Línea de referencia en 50%
ax.axvline(50, color='tomato', linestyle='--', linewidth=1.2, alpha=0.7, label='50% de faltantes')

ax.set_title('Porcentaje de valores faltantes en consumo_total_mixto\npor alcaldía', fontsize=13)
ax.set_xlabel('% de registros faltantes', fontsize=11)
ax.set_ylabel('')
ax.legend()

# Etiquetas con el % al final de cada barra
for i, (idx, row) in enumerate(resumen_miss.iterrows()):
    ax.text(row['pct_faltante'] + 0.5, i, f"{row['pct_faltante']:.1f}%", va='center', fontsize=9)

plt.tight_layout()
plt.show()

**¿Qué nos dice este gráfico?**
- Los faltantes **no son aleatorios**: algunas alcaldías tienen casi el 100% de faltantes en consumo mixto, mientras otras no tienen ninguno.
- Esto sugiere que el consumo mixto simplemente no existe (o no se registra) en ciertas alcaldías — no es un error de captura, sino una diferencia real en el tipo de uso.

## 6. Errores frecuentes en visualización

Antes de continuar, conviene revisar los errores más comunes que hacen los gráficos difíciles de leer (o peor: engañosos).

### Error 1: Eje Y que no empieza en cero

Cuando el eje Y empieza en un valor distinto de cero, las diferencias entre barras se ven **exageradas**.

In [ ]:
prom_bim = df.groupby('bimestre')['consumo_total'].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Malo: eje Y recortado — las diferencias parecen enormes
prom_bim.plot(kind='bar', ax=axes[0], color='tomato', edgecolor='white')
axes[0].set_ylim(prom_bim.min() * 0.95, prom_bim.max() * 1.05)  # eje recortado
axes[0].set_title('Eje Y recortado\n(diferencias exageradas)', fontsize=12)
axes[0].set_xlabel('Bimestre')
axes[0].set_ylabel('Consumo promedio (m³)')
axes[0].tick_params(axis='x', rotation=0)

# Bueno: eje Y desde cero — diferencias reales
prom_bim.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_ylim(0, prom_bim.max() * 1.15)  # eje desde cero
axes[1].set_title('Eje Y desde cero\n(diferencias en perspectiva)', fontsize=12)
axes[1].set_xlabel('Bimestre')
axes[1].set_ylabel('Consumo promedio (m³)')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('El mismo dato, dos percepciones diferentes', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Error 2: Etiquetas del eje X rotadas e ilegibles

In [ ]:
prom_alc = df.groupby('alcaldia')['consumo_total'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Malo: barras verticales con nombres largos rotados
prom_alc.plot(kind='bar', ax=axes[0], color='tomato', edgecolor='white')
axes[0].set_title('Barras verticales con nombres largos', fontsize=12)
axes[0].set_ylabel('Consumo promedio (m³)')
axes[0].tick_params(axis='x', rotation=45)

# Bueno: barras horizontales — nombres legibles sin rotar
prom_alc.sort_values().plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Barras horizontales — legible sin rotar', fontsize=12)
axes[1].set_xlabel('Consumo promedio (m³)')

plt.tight_layout()
plt.show()

### Error 3: Demasiadas variables en una sola gráfica

In [ ]:
# Malo: 8 líneas en una sola gráfica — no se puede leer
cols_consumo_todas = ['consumo_total', 'consumo_total_dom', 'consumo_total_no_dom',
                      'consumo_prom', 'consumo_prom_dom', 'consumo_prom_no_dom',
                      'consumo_total_mixto', 'consumo_prom_mixto']

medias_bim = df.groupby('bimestre')[cols_consumo_todas].mean()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Mal: todo junto
for col in cols_consumo_todas:
    axes[0].plot(medias_bim.index, medias_bim[col], marker='o', label=col)
axes[0].set_title('Demasiadas líneas — ilegible', fontsize=12)
axes[0].legend(fontsize=7)
axes[0].set_xlabel('Bimestre')
axes[0].set_ylabel('m³')

# Bien: solo las variables de consumo total
cols_total = ['consumo_total_dom', 'consumo_total_no_dom', 'consumo_total_mixto']
etiquetas  = ['Doméstico', 'No doméstico', 'Mixto']
colores_lín = ['steelblue', 'coral', 'mediumseagreen']

for col, etiq, color in zip(cols_total, etiquetas, colores_lín):
    axes[1].plot(medias_bim.index, medias_bim[col], marker='o', label=etiq, color=color, linewidth=2)

axes[1].set_title('Solo las variables relevantes', fontsize=12)
axes[1].set_xlabel('Bimestre')
axes[1].set_ylabel('Consumo promedio (m³)')
axes[1].legend(title='Tipo de uso')

plt.tight_layout()
plt.show()

## 7. Gráfica de línea — Variables temporales

**¿Cuándo usarla?** Para mostrar cómo cambia algo **en el tiempo**. Las líneas implican continuidad — que hay un orden entre los puntos. Por eso solo se usan con variables temporales u ordinales.

**Regla:** Nunca uses una gráfica de línea para comparar categorías sin orden (como alcaldías). En ese caso usa barras.

### ¿Cómo evoluciona el consumo por tipo de uso a lo largo de los bimestres?

### 7.1 Consumo total por índice de desarrollo a través del tiempo

In [ ]:
# Agrupar por fecha de registro e índice de desarrollo y sumar el consumo total
consumo_fecha_indice = df.groupby(['fecha_registro', 'indice_des'])['consumo_total'].sum().reset_index()

# Definimos el orden lógico para la leyenda
orden_indice_plot = ['alto', 'medio', 'bajo', 'popular']

fig, ax = plt.subplots(figsize=(12, 6))

sns.lineplot(
    data=consumo_fecha_indice,
    x='fecha_registro',
    y='consumo_total',
    hue='indice_des',
    palette=paleta, # Reutilizamos la paleta de colores definida anteriormente
    hue_order=orden_indice_plot, # Aseguramos el orden de las líneas en la leyenda
    #marker='o', # Añadimos marcadores para cada punto de datos
    linewidth=1.5,
    ax=ax
)

ax.set_title('Consumo Total de Agua por Índice de Desarrollo a través del Tiempo', fontsize=14)
ax.set_xlabel('Fecha de Registro', fontsize=12)
ax.set_ylabel('Consumo Total (m³)', fontsize=12)
ax.legend(title='Índice de Desarrollo', loc='upper left')
ax.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# Agrupar por semana e índice de desarrollo y calcular el promedio semanal del consumo total
consumo_semanal_indice = df.groupby([pd.Grouper(key='fecha_registro', freq='W'), 'indice_des'])['consumo_total'].mean().reset_index()

# Definimos el orden lógico para la leyenda
orden_indice_plot = ['alto', 'medio', 'bajo', 'popular']

fig, ax = plt.subplots(figsize=(12, 6))

sns.lineplot(
    data=consumo_semanal_indice,
    x='fecha_registro',
    y='consumo_total',
    hue='indice_des',
    palette=paleta, # Reutilizamos la paleta de colores definida anteriormente
    hue_order=orden_indice_plot, # Aseguramos el orden de las líneas en la leyenda
    marker='o', # Añadimos marcadores para cada punto de datos
    linewidth=1.5,
    ax=ax
)

ax.set_title('Consumo Promedio Semanal de Agua por Índice de Desarrollo a través del Tiempo', fontsize=14)
ax.set_xlabel('Fecha de Registro (Semanal)', fontsize=12)
ax.set_ylabel('Consumo Promedio Semanal (m³)', fontsize=12)
ax.legend(title='Índice de Desarrollo', loc='upper left')
ax.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# ¿Qué hace esta línea?
import matplotlib.ticker as mticker

# Seleccionar las columnas de consumo y agrupar por fecha de registro
#consumo_por_tipo = df.groupby('fecha_registro')[['consumo_total_mixto', 'consumo_total_dom', 'consumo_total_no_dom']].sum().reset_index()

# Agrupar por semana y calcular el promedio semanal del consumo por tipo
consumo_por_tipo = df.groupby(pd.Grouper(key='fecha_registro', freq='W'))[['consumo_total_mixto', 'consumo_total_dom', 'consumo_total_no_dom']].mean().reset_index()

# Transformar los datos a formato 'long' para seaborn
consumo_long = pd.melt(consumo_por_tipo,
                         id_vars=['fecha_registro'],
                         var_name='Tipo de Consumo',
                         value_name='Consumo Total')

# Mapeo para etiquetas más legibles y colores consistentes
mapeo_nombres = {
    'consumo_total_mixto': 'Mixto',
    'consumo_total_dom': 'Doméstico',
    'consumo_total_no_dom': 'No Doméstico'
}
consumo_long['Tipo de Consumo'] = consumo_long['Tipo de Consumo'].map(mapeo_nombres)

colores_lineas = {
    'Doméstico': 'steelblue',
    'No Doméstico': 'coral',
    'Mixto': 'mediumseagreen'
}

fig, ax = plt.subplots(figsize=(12, 6))

sns.lineplot(
    data=consumo_long,
    x='fecha_registro',
    y='Consumo Total',
    hue='Tipo de Consumo',
    palette=colores_lineas,
    #marker='o',
    linewidth=1.5,
    ax=ax
)

ax.set_title('Consumo Total de Agua por Tipo de Uso a través del Tiempo', fontsize=14)
ax.set_xlabel('Fecha de Registro', fontsize=12)
ax.set_ylabel('Consumo Total (m³)', fontsize=12)
ax.ticklabel_format(axis='y', style='plain', useOffset=False, useLocale=True) # Apply comma separator to y-axis
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(title='Tipo de Uso', loc='upper left')
ax.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

> ¿Notas algo sobre el tiempo de esta variable?

## 8. Checklist antes de entregar una gráfica

Antes de incluir cualquier gráfico en un reporte o presentación, revisa esta lista:

```
☐  ¿El tipo de gráfico es el correcto para las variables que estoy mostrando?

☐  ¿Tiene título? ¿El título dice qué muestra el gráfico, no solo el nombre de la variable?

☐  ¿Los ejes tienen etiquetas con unidades?

☐  ¿El eje Y empieza en cero? (Para barras: siempre. Para líneas: casi siempre.)

☐  ¿Las categorías están ordenadas de forma lógica? (no alfabética)

☐  ¿Se puede leer sin rotar la cabeza? (Si hay etiquetas rotadas, considera barras horizontales)

☐  Si recorté outliers para hacer el gráfico más legible, ¿lo indico en el título o en una nota?

☐  ¿La leyenda es necesaria? ¿Está en un lugar que no tape los datos?

☐  ¿Los colores son distinguibles? (Evitar rojo-verde juntos; son indistinguibles para daltónicos)

☐  ¿Hay una conclusión o interpretación acompañando el gráfico?

☐  ¿Sí los ejes están en números mayores a mil, tienen coma?

```
